In [1]:
import pandas as pd

movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

movies = movies.merge(credits, on='title')

In [2]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [3]:
movies.dropna(inplace=True)

In [4]:
import ast

def convert(text):
    L = []

    for i in ast.literal_eval(text):
        L.append(i['name'])

    return L

In [5]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [6]:
def convert_cast(text):

    L = []
    counter = 0

    for i in ast.literal_eval(text):

        if counter != 3:
            L.append(i['name'])
            counter += 1

        else:
            break

    return L

movies['cast'] = movies['cast'].apply(convert_cast)

In [7]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [8]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])

movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])

movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])

movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [9]:
movies['tags'] = (
    movies['overview'] +
    movies['genres'] +
    movies['keywords'] +
    movies['cast'] +
    movies['crew']
)

In [10]:
new_df = movies[['movie_id','title','tags']]

In [11]:
new_df = movies[['movie_id','title','tags']].copy()

new_df['tags'] = new_df['tags'].apply(
    lambda x: " ".join(x)
)

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = cv.fit_transform(new_df['tags']).toarray()

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [14]:
import pickle
import os

os.makedirs('../model', exist_ok=True)

pickle.dump(
    new_df,
    open('../model/movies.pkl', 'wb')
)

pickle.dump(
    similarity,
    open('../model/similarity.pkl', 'wb')
)

In [15]:
def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x:x[1]
    )[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [16]:
recommend('Avatar')

Titan A.E.
Small Soldiers
Independence Day
Aliens vs Predator: Requiem
Ender's Game


In [17]:
import pickle

pickle.dump(new_df, open('../model/movies.pkl','wb'))

pickle.dump(similarity, open('../model/similarity.pkl','wb'))